In [1]:
import pandas as pd
import numpy as np
from path_config import PathConfig
paths = PathConfig()

In [2]:
dl_data = pd.read_excel(paths.dl_path, sheet_name='Series Formatted Data')
dl_data = dl_data.drop(["Message","Technology_Mode","NR_UE_Timing_Advance","NR_UE_Power_Tx_PRACH_0"],axis=1)
dl_data = dl_data.loc[:, ~dl_data.columns.str.contains('^Unnamed')]

kabinets = pd.read_csv((paths._processed_saha_olcum_dir / "Kabinets.csv"))
pci_cols = list(kabinets["PCI"])

In [3]:
def dl_group_by_col(data, columns):

    if isinstance(columns, str):
        columns = [columns]
    
    new_data = pd.DataFrame()

    last_known_data_cols = ["Time",'Longitude', 'Latitude','NR_UE_PCI_0','NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_PCI_1',
                           'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_PCI_4',
                           'NR_UE_Pathloss_DL_0', 'App_Throughput_DL',
                           'NR_UE_Modulation_Avg_DL_0', 'NR_UE_RI_DL_0', 
                           'NR_UE_CCE_AggregationLev_0', 'NR_UE_Power_Tx_PUSCH_0'] 
    
    mean_cols = ['NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0','NR_UE_Nbr_RSRP_0', 'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRP_2','NR_UE_RB_Num_DL_0',
                'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_0','NR_UE_Nbr_RSRQ_1', 'NR_UE_BLER_DL_0','NR_UE_Nbr_RSRQ_2', 'NR_UE_Nbr_RSRQ_3',
                'NR_UE_Nbr_RSRQ_4', 'NR_UE_Ack_As_Nack_DL_0','NR_UE_NACK_Rate_DL_0','NR_UE_Throughput_PDCP_DL', 'NR_UE_NACK_Rate_UL_0']
    """
    "NR_UE_NACK_Rate_UL_0",
    "NR_UE_Throughput_PDCP_DL",
    "NR_UE_NACK_Rate_DL_0",
    "NR_UE_Ack_As_Nack_DL_0",
    "NR_UE_RB_Num_DL_0",
    "NR_UE_BLER_DL_0"
    """
    for column in columns:
        if column in last_known_data_cols:
            last_known_data_cols.remove(column)
        if column in mean_cols:
            mean_cols.remove(column)
    
    unique_groups = data.groupby(columns).size().reset_index()[columns]
    
    for idx, group_values in unique_groups.iterrows():
        group_filter = True
        for col in columns:
            group_filter = group_filter & (data[col] == group_values[col])
        
        group_data = data[group_filter]
        
        filtered_row_mean = group_data[mean_cols].mean()
        filtered_row_last = group_data[last_known_data_cols]
        
        last_known_values = filtered_row_last.apply(lambda x: x.dropna().iloc[-1] if not x.dropna().empty else np.nan)
        
        group_series = pd.Series(group_values)
        filtered_row = pd.concat([group_series, filtered_row_mean, last_known_values])
        new_data = pd.concat([new_data, filtered_row.to_frame().T], ignore_index=True)
    
    return new_data

In [4]:
data = dl_group_by_col(dl_data, columns=["Longitude","Latitude"])
data

,Longitude,Latitude,NR_UE_RSRP_0,NR_UE_RSRQ_0,NR_UE_SINR_0,NR_UE_Nbr_RSRP_0,NR_UE_Nbr_RSRP_1,NR_UE_Nbr_RSRP_2,NR_UE_RB_Num_DL_0,NR_UE_Nbr_RSRP_3,...,NR_UE_Nbr_PCI_1,NR_UE_Nbr_PCI_2,NR_UE_Nbr_PCI_3,NR_UE_Nbr_PCI_4,NR_UE_Pathloss_DL_0,App_Throughput_DL,NR_UE_Modulation_Avg_DL_0,NR_UE_RI_DL_0,NR_UE_CCE_AggregationLev_0,NR_UE_Power_Tx_PUSCH_0
0,29.01534,41.10568,-124.3,-17.35,-9.1,-125.15,-124.5,NaN,121.0,NaN,...,76.0,NaN,NaN,NaN,151.7,26372.78,16QAM,Rank1,LEVEL_16,22.2
1,29.01534,41.10574,-125.0,-18.05,-8.55,-123.3,-123.05,-123.8,124.5,NaN,...,40.0,76.0,NaN,NaN,147.2,23868.11,16QAM,Rank1,LEVEL_16,22.2
2,29.01534,41.10579,-125.35,-18.75,-10.5,-124.9,-124.15,-122.5,123.0,-125.4,...,30.0,40.0,76.0,NaN,147.8,32195.62,16QAM,Rank1,LEVEL_16,22.2
3,29.01534,41.10581,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
4,29.01534,41.10583,-123.65,-18.1,-9.55,-124.75,-122.6,-121.2,162.5,NaN,...,76.0,76.0,NaN,NaN,143.3,33435.66,16QAM,Rank1,LEVEL_16,22.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,29.03115,41.09979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
1510,29.03116,41.09979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
1511,29.03116,41.09983,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT
1512,29.03117,41.09979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT,NaT


In [5]:
connected_pci_cols = ["NR_UE_PCI_0", "NR_UE_Nbr_PCI_0", "NR_UE_Nbr_PCI_1", "NR_UE_Nbr_PCI_2", "NR_UE_Nbr_PCI_3", "NR_UE_Nbr_PCI_4"]
invalid_pci_cols = []
for col in connected_pci_cols:
    unique_pci_cols = [int(pci) for pci in dl_data["NR_UE_PCI_0"].dropna().unique() if pci not in pci_cols]
    invalid_pci_cols.extend(unique_pci_cols)

invalid_pci_cols = list(set(invalid_pci_cols))    
invalid_pci_cols
data[connected_pci_cols] = data[connected_pci_cols].replace(invalid_pci_cols, np.nan)

In [6]:
forward_fill_cols = [
    "NR_UE_PCI_0",
    "NR_UE_CCE_AggregationLev_0",
]

linear_cols = [
    "NR_UE_Pathloss_DL_0",
    "NR_UE_RSRP_0",
    "NR_UE_RSRQ_0",
    "NR_UE_SINR_0",
    "NR_UE_NACK_Rate_UL_0",
    "NR_UE_Throughput_PDCP_DL",
    "NR_UE_NACK_Rate_DL_0",
    "NR_UE_Ack_As_Nack_DL_0",
    "NR_UE_RB_Num_DL_0",
    "NR_UE_BLER_DL_0",
    "NR_UE_Power_Tx_PUSCH_0",
    "App_Throughput_DL"
]

In [7]:
"""
for col in forward_fill_cols:
    if col in data.columns:
        data[col] = data[col].fillna(method='ffill')

for col in forward_fill_cols:
    if col in data.columns:
        data[col] = data[col].fillna(method='bfill')
"""
for col in linear_cols:
    if col in data.columns:
        # String'leri sayıya çevir, hataları NaN yap
        data[col] = pd.to_numeric(data[col], errors='coerce')
        
data[linear_cols] = data[linear_cols].interpolate(method='linear', limit_direction="both")

In [8]:
# RI değeri, o anda 5G bağlantısının kaç katmanlı MIMO kullandığını gösterir. 
# Yüksek değer = daha iyi kanal koşulları ve potansiyel olarak daha yüksek veri hızı.
# NR_UE_RI_DL_0 sütunu SINR değerlerine göre dolduruluyor.
# SINR sütununda boş veri varken çalışmamalı.

def rank_fill(df, referance_column, target_column):
    ranks = df[target_column].dropna().unique()
    mean_ranks  = {rank:df[df[target_column]==rank][referance_column].mean() for rank in ranks}
    null_mask = df[target_column].isnull()

    for rank in range(len(ranks)-1):
        condition = (null_mask & (df[referance_column] < (mean_ranks[ranks[rank]] + mean_ranks[ranks[rank+1]])/2))
        
        df.loc[condition, target_column] = ranks[rank+1]

    df.loc[df[target_column].isnull(), target_column] = ranks[-1]
    return df

In [9]:
def fill_ffill_bfill(df, forward_fill_cols):    
    for col in forward_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(method='ffill')

    for col in forward_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(method='bfill')
    return df

In [10]:
data = rank_fill(data,referance_column = "NR_UE_SINR_0", target_column = "NR_UE_RI_DL_0" )
data = rank_fill(data,referance_column = "NR_UE_SINR_0", target_column = "NR_UE_Modulation_Avg_DL_0" )

In [11]:
data = data.set_index('Time').sort_index()
data["NR_UE_PCI_0"] = pd.to_numeric(data["NR_UE_PCI_0"], errors='coerce')
data["NR_UE_PCI_0"] = data["NR_UE_PCI_0"].interpolate(method='nearest')
data = data.reset_index()

/Users/kutaydogan/Documents/5G Teknofest/5G-Positioning-Competition/.venv/lib/python3.10/site-packages/pandas/core/indexes/base.py:7588: FutureWarning: Dtype inference on a pandas object (Series, Index, ExtensionArray) is deprecated. The Index constructor will keep the original dtype in the future. Call `infer_objects` on the result to get the old behavior.
  return Index(sequences[0], name=names)


In [12]:
data = fill_ffill_bfill(data, forward_fill_cols)

/var/folders/98/pjzr794d191gcyxknpbptych0000gn/T/ipykernel_20624/1438752442.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill')
/var/folders/98/pjzr794d191gcyxknpbptych0000gn/T/ipykernel_20624/1438752442.py:8: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='bfill')


In [13]:
def create_signal_data(signal_column, pci_column, data, col_names, fill_na_value=-1000):
    #col_names = np.append(col_names, ['Longitude','Latitude'])
    signal_data_list = []
    for idx, row in data[pci_column].iterrows():
        
        temp_dict = {}
        for t,i in enumerate(row):
            if not np.isnan(i):
                temp_dict[int(i)] = data[signal_column[t]][idx]
                #temp_dict['Longitude'], temp_dict['Latitude'] = data['Longitude'][idx], data['Latitude'][idx]
        signal_data_list.append(temp_dict)
    signal_data = pd.DataFrame(signal_data_list)[col_names].fillna(fill_na_value)

    
    return signal_data

pci_cols = connected_pci_cols

data[pci_cols] = data[pci_cols].replace({pd.NaT: np.nan})
data[pci_cols] = data[pci_cols].astype(float)

rsrp_cols = [col for col in data.columns if "RSRP" in col]
rsrq_cols = [col for col in data.columns if "RSRQ" in col]


signal_data = create_signal_data(
    signal_column = rsrp_cols, 
    pci_column = pci_cols, 
    data = data, 
    col_names = kabinets["PCI"].values
    )

signal_data = signal_data.add_prefix("RSRP_PCI_")
data = data.join(signal_data, how="left")

signal_data = create_signal_data(
    signal_column = rsrq_cols, 
    pci_column = pci_cols, 
    data = data, 
    col_names = kabinets["PCI"].values
    )

signal_data = signal_data.add_prefix("RSRQ_PCI_")
data = data.join(signal_data, how="left")

data = data.drop(columns=['NR_UE_PCI_0','NR_UE_Nbr_PCI_0', 'NR_UE_Nbr_PCI_1',
                           'NR_UE_Nbr_PCI_2', 'NR_UE_Nbr_PCI_3', 'NR_UE_Nbr_PCI_4',
                           'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_Nbr_RSRP_0', 
                           'NR_UE_Nbr_RSRP_1', 'NR_UE_Nbr_RSRP_2','NR_UE_Nbr_RSRQ_2','NR_UE_Nbr_RSRQ_3','NR_UE_Nbr_RSRQ_4',
                            'NR_UE_Nbr_RSRP_3', 'NR_UE_Nbr_RSRP_4', 'NR_UE_Nbr_RSRQ_0','NR_UE_Nbr_RSRQ_1',])

/var/folders/98/pjzr794d191gcyxknpbptych0000gn/T/ipykernel_20624/1198395317.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data[pci_cols] = data[pci_cols].replace({pd.NaT: np.nan})


In [15]:
data= pd.get_dummies(data,columns=["NR_UE_CCE_AggregationLev_0",'NR_UE_Modulation_Avg_DL_0','NR_UE_RI_DL_0'])

In [16]:
distance = 10**((data['NR_UE_Pathloss_DL_0'] - 20*np.log10(3.5*1e9)+147.55)/20)
data["Distance"] = distance

In [18]:
data.isnull().sum()

Time                                   0
Longitude                              0
Latitude                               0
NR_UE_SINR_0                           0
NR_UE_RB_Num_DL_0                      0
NR_UE_BLER_DL_0                        0
NR_UE_Ack_As_Nack_DL_0                 0
NR_UE_NACK_Rate_DL_0                   0
NR_UE_Throughput_PDCP_DL               0
NR_UE_NACK_Rate_UL_0                   0
NR_UE_Pathloss_DL_0                    0
App_Throughput_DL                      0
NR_UE_Power_Tx_PUSCH_0                 0
RSRP_PCI_30                            0
RSRP_PCI_40                            0
RSRP_PCI_59                            0
RSRP_PCI_48                            0
RSRP_PCI_68                            0
RSRP_PCI_76                            0
RSRP_PCI_3                             0
RSRP_PCI_13                            0
RSRP_PCI_23                            0
RSRQ_PCI_30                            0
RSRQ_PCI_40                            0
RSRQ_PCI_59     

In [19]:
data.to_csv(paths._processed_saha_olcum_dir / "dl_data_group_by_location.csv")

In [22]:
data.columns

Index(['Time', 'Longitude', 'Latitude', 'NR_UE_SINR_0', 'NR_UE_RB_Num_DL_0',
       'NR_UE_BLER_DL_0', 'NR_UE_Ack_As_Nack_DL_0', 'NR_UE_NACK_Rate_DL_0',
       'NR_UE_Throughput_PDCP_DL', 'NR_UE_NACK_Rate_UL_0',
       'NR_UE_Pathloss_DL_0', 'App_Throughput_DL', 'NR_UE_Power_Tx_PUSCH_0',
       'RSRP_PCI_30', 'RSRP_PCI_40', 'RSRP_PCI_59', 'RSRP_PCI_48',
       'RSRP_PCI_68', 'RSRP_PCI_76', 'RSRP_PCI_3', 'RSRP_PCI_13',
       'RSRP_PCI_23', 'RSRQ_PCI_30', 'RSRQ_PCI_40', 'RSRQ_PCI_59',
       'RSRQ_PCI_48', 'RSRQ_PCI_68', 'RSRQ_PCI_76', 'RSRQ_PCI_3',
       'RSRQ_PCI_13', 'RSRQ_PCI_23', 'NR_UE_CCE_AggregationLev_0_LEVEL_1',
       'NR_UE_CCE_AggregationLev_0_LEVEL_16',
       'NR_UE_CCE_AggregationLev_0_LEVEL_2',
       'NR_UE_CCE_AggregationLev_0_LEVEL_4',
       'NR_UE_CCE_AggregationLev_0_LEVEL_8', 'NR_UE_Modulation_Avg_DL_0_16QAM',
       'NR_UE_Modulation_Avg_DL_0_256QAM', 'NR_UE_Modulation_Avg_DL_0_64QAM',
       'NR_UE_Modulation_Avg_DL_0_QPSK', 'NR_UE_RI_DL_0_Rank1',
       'NR_U